In [ ]:
### Regions were generated using 'get_geneBodies.ipynb'

In [15]:
#### Fig. 6b

import bbi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42 
matplotlib.rcParams['svg.fonttype'] = 'none' 

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'], 
    'font.size': 9,     
    'axes.labelsize': 9,  
    'xtick.labelsize': 7,     
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.titlesize': 9,
})

########## config
BW = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_mega120mPD_DpnII_R12_20250129_5kb.bw"

bed_sets = {   # label -> path
    "speckle": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_expressed_10kbfilter2.bed",
    "speckle_low": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_silent_10kbfilter2.bed",
    "Act23": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_InteriorAct23_expressed_10kbfilter2.bed",
    "Act23_low": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_InteriorAct23_silent_10kbfilter2.bed",
}
colors = {
    "speckle":      "#9E0142",  
    "speckle_low":  "#F1A9A0", 
    "Act23":         "#F46D43", 
    "Act23_low":     "#FDAE61",  
}

########### scaled-mode geometry 
upstream   = 10000    
downstream = 10000  
body_bins  = 100    
flank_bins = 100   

half = 5   

def stackup_scaled(f, reg, bw_chroms):
    """Return (n_genes x total_bins) matrix in scaled-regions mode, strand-oriented."""
    reg = reg[reg.chrom.isin(bw_chroms)].reset_index(drop=True)

    chrom  = reg.chrom.values
    start  = reg.start.values          
    end    = reg.end.values
    strand = reg.strand.values

    # genomic anchor points (TSS / TES in genomic space)
    # for +: TSS=start, TES=end ; for -: TSS=end, TES=start
    is_plus = strand == "+"

    #  upstream flank (genomic, before TSS) 
    up_start = np.where(is_plus, start - upstream, end)
    up_end   = np.where(is_plus, start,            end + upstream)
    up = f.stackup(chrom, up_start, up_end, bins=flank_bins,
                   missing=np.nan, oob=np.nan)

    #  gene body (TSS->TES, rescaled to body_bins) 
    body = f.stackup(chrom, start, end, bins=body_bins,
                     missing=np.nan, oob=np.nan)

    #  downstream flank (genomic, after TES) 
    dn_start = np.where(is_plus, end,              start - downstream)
    dn_end   = np.where(is_plus, end + downstream, start)
    dn = f.stackup(chrom, dn_start, dn_end, bins=flank_bins,
                   missing=np.nan, oob=np.nan)

    #  flip minus-strand rows so all run 5'->3' left to right 
    minus = ~is_plus
    up[minus]   = up[minus][:, ::-1]
    body[minus] = body[minus][:, ::-1]
    dn[minus]   = dn[minus][:, ::-1]

    # concatenate: [upstream | body | downstream]
    return np.hstack([up, body, dn])

######## build stackups
stacks, profiles = {}, {}
with bbi.open(BW) as f:
    bw_chroms = set(f.chromsizes)
    for label, path in bed_sets.items():
        reg = pd.read_csv(path, sep="\t", header=None,
                          names=["chrom", "start", "end", "enst",
                                 "ensg", "strand", "length"])
        s = stackup_scaled(f, reg, bw_chroms)

        # drop empty rows
        s = s[~np.isnan(s).all(axis=1)]

        # sort by center-of-body signal
        # --- sort by signal AT THE TSS (boundary between flank and body) ---
        tss_col = flank_bins                         # first body column = TSS
        half    = 5                                   # bins each side of TSS
        t0 = max(0, tss_col - half)
        t1 = tss_col + half
        
        with np.errstate(invalid="ignore"):
            key = np.nanmean(s[:, t0:t1], axis=1)
        key = np.where(np.isnan(key), -np.inf, key)
        s   = s[np.argsort(key)[::-1]]

        stacks[label]   = s
        profiles[label] = np.nanmean(s, axis=0)

######## shared color scale
allvals = np.concatenate([s[np.isfinite(s)].ravel() for s in stacks.values()])
vmin, vmax = np.nanpercentile(allvals, [2, 98])

######## x-axis tick positions for TSS / TES
total_bins = flank_bins + body_bins + flank_bins
tss_pos = flank_bins                         # boundary upstream|body
tes_pos = flank_bins + body_bins             # boundary body|downstream

######## figure
labels = list(stacks.keys())
n = len(labels)
row_counts = np.array([len(stacks[l]) for l in labels], dtype=float)
rel = np.maximum(row_counts / row_counts.max(), 0.15)
heights = [0.64] + list(rel)
fig_h = 2 + 2.2 * n

fig, axes = plt.subplots(
    n + 1, 1, figsize=(2.5, fig_h),
    height_ratios=heights, gridspec_kw={"hspace": 0.105}, sharex=True,
)

# profile
ax_prof = axes[0]
xx = np.arange(total_bins)
for label in labels:
    ax_prof.plot(xx, profiles[label], color=colors[label], lw=1)
ax_prof.set_ylabel("Mean LOS\nresidual")
ax_prof.margins(x=0)
for xb in (tss_pos, tes_pos):
    ax_prof.axvline(xb, color="grey", lw=0.6, ls="--")

# heatmaps
im = None
for i, label in enumerate(labels):
    ax = axes[i + 1]
    im = ax.imshow(
        stacks[label], aspect="auto", cmap="magma",
        vmin=vmin, vmax=vmax, interpolation="none",
        extent=[0, total_bins, 0, len(stacks[label])],
    )
    ax.set_ylabel(f"{label}\n(n={len(stacks[label])})",
                  rotation=0, ha="right", va="center", fontsize=9)
    ax.set_yticks([])
    for xb in (tss_pos, tes_pos):
        ax.axvline(xb, color="white", lw=0.2, ls="--", alpha=0.7)
    for spine in ax.spines.values():
        spine.set_edgecolor(colors[label]); spine.set_linewidth(2); spine.set_visible(True)
    if i < n - 1:
        ax.tick_params(labelbottom=False)

# x ticks: TSS / TES labels
axes[-1].set_xticks([0, tss_pos, tes_pos, total_bins])
axes[-1].set_xticklabels([f"-{upstream//1000}kb", "TSS", "TES",
                          f"+{downstream//1000}kb"])
for spine in ax.spines.values():
    spine.set_linewidth(1.3)
ax.tick_params(width=1.3, length=2)

# shared colorbar
fig.subplots_adjust(right=0.86)
cax = fig.add_axes([0.88, 0.15, 0.05, 0.6])
fig.colorbar(im, cax=cax, label="LOS residual")

plt.savefig("gene_scaled_stackups_Act23_new_regions2.svg", bbox_inches="tight")
plt.close(fig)

/tmp/ipykernel_1551950/2654477332.py:105: RuntimeWarning: Mean of empty slice
  key = np.nanmean(s[:, t0:t1], axis=1)
/tmp/ipykernel_1551950/2654477332.py:105: RuntimeWarning: Mean of empty slice
  key = np.nanmean(s[:, t0:t1], axis=1)
/tmp/ipykernel_1551950/2654477332.py:105: RuntimeWarning: Mean of empty slice
  key = np.nanmean(s[:, t0:t1], axis=1)
/tmp/ipykernel_1551950/2654477332.py:105: RuntimeWarning: Mean of empty slice
  key = np.nanmean(s[:, t0:t1], axis=1)
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
fin

In [16]:
### Fig. 6c

import bbi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42 
matplotlib.rcParams['svg.fonttype'] = 'none' t

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'], 
    'font.size': 9,            
    'axes.labelsize': 9,     
    'xtick.labelsize': 7,       
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.titlesize': 9,
})

######### config
BW_LOS = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_mega120mPD_DpnII_R12_20250129_5kb.bw"
BW_NEW = "/abyss/dlafonta/data/K562_data/DL/POLR2A_R1_ENCFF496FVA.bigWig"        

bed_sets = {   # label -> path
    "speckle": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_expressed_10kbfilter2.bed",
    "speckle_low": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_silent_10kbfilter2.bed",
    "Act23": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_InteriorAct23_expressed_10kbfilter2.bed",
    "Act23_low": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_InteriorAct23_silent_10kbfilter2.bed",
}
colors = {
    "speckle":      "#9E0142",
    "speckle_low":  "#F1A9A0",  
    "Act23":         "#F46D43", 
    "Act23_low":     "#FDAE61",  
}

upstream = downstream = 10000
body_bins, flank_bins = 50, 50
half = 5

total_bins = flank_bins + body_bins + flank_bins
tss_pos, tes_pos = flank_bins, flank_bins + body_bins

def stackup_scaled(f, reg, bw_chroms):
    reg = reg[reg.chrom.isin(bw_chroms)].reset_index(drop=True)
    chrom, start, end = reg.chrom.values, reg.start.values, reg.end.values
    is_plus = reg.strand.values == "+"

    up_start = np.where(is_plus, start - upstream, end)
    up_end   = np.where(is_plus, start,            end + upstream)
    up = f.stackup(chrom, up_start, up_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    body = f.stackup(chrom, start, end, bins=body_bins, missing=np.nan, oob=np.nan)

    dn_start = np.where(is_plus, end,              start - downstream)
    dn_end   = np.where(is_plus, end + downstream, start)
    dn = f.stackup(chrom, dn_start, dn_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    minus = ~is_plus
    up[minus]   = up[minus][:, ::-1]
    body[minus] = body[minus][:, ::-1]
    dn[minus]   = dn[minus][:, ::-1]
    return np.hstack([up, body, dn])


####### PASS 1: build LOS stackups, determine sort order + valid rows

orders = {}      # label -> row index order (into the valid-filtered matrix)
valids = {}      # label -> boolean mask of non-empty rows (into full reg)
los_stacks = {}

with bbi.open(BW_LOS) as f:
    bw_chroms = set(f.chromsizes)
    for label, path in bed_sets.items():
        reg = pd.read_csv(path, sep="\t", header=None,
                          names=["chrom","start","end","enst","ensg","strand","length"])
        s = stackup_scaled(f, reg, bw_chroms)

        valid = ~np.isnan(s).all(axis=1)        # which rows survive
        s = s[valid]

        t0, t1 = max(0, tss_pos - half), tss_pos + half
        with np.errstate(invalid="ignore"):
            key = np.nanmean(s[:, t0:t1], axis=1)
        key = np.where(np.isnan(key), -np.inf, key)
        order = np.argsort(key)[::-1]

        valids[label] = valid
        orders[label] = order
        los_stacks[label] = s[order]


# PASS 2: build NEW-track stackups, apply SAME valid mask + order

def build_with_fixed_order(bw_path):
    out = {}
    with bbi.open(bw_path) as f:
        bw_chroms = set(f.chromsizes)
        for label, path in bed_sets.items():
            reg = pd.read_csv(path, sep="\t", header=None,
                              names=["chrom","start","end","enst","ensg","strand","length"])
            s = stackup_scaled(f, reg, bw_chroms)
            s = s[valids[label]]                 # SAME rows dropped as LOS pass
            s = s[orders[label]]                 # SAME order as LOS pass
            out[label] = s
    return out

new_stacks = build_with_fixed_order(BW_NEW)
profiles   = {l: np.nanmean(s, axis=0) for l, s in new_stacks.items()}

/tmp/ipykernel_1551950/3244388949.py:85: RuntimeWarning: Mean of empty slice
  key = np.nanmean(s[:, t0:t1], axis=1)
/tmp/ipykernel_1551950/3244388949.py:85: RuntimeWarning: Mean of empty slice
  key = np.nanmean(s[:, t0:t1], axis=1)
/tmp/ipykernel_1551950/3244388949.py:85: RuntimeWarning: Mean of empty slice
  key = np.nanmean(s[:, t0:t1], axis=1)
/tmp/ipykernel_1551950/3244388949.py:85: RuntimeWarning: Mean of empty slice
  key = np.nanmean(s[:, t0:t1], axis=1)


In [17]:
# 1. repoint stacks/profiles to the NEW track
stacks   = new_stacks
profiles = {l: np.nanmean(s, axis=0) for l, s in stacks.items()}

# 2. recompute the color scale from the NEW data
allvals = np.concatenate([s[np.isfinite(s)].ravel() for s in stacks.values()])
vmin = 0
vmax = np.nanpercentile(allvals, 99)

###### PLot figure

labels = list(stacks.keys())
n = len(labels)
row_counts = np.array([len(stacks[l]) for l in labels], dtype=float)
rel = np.maximum(row_counts / row_counts.max(), 0.15)
heights = [0.64] + list(rel)
fig_h = 2 + 2.2 * n

fig, axes = plt.subplots(
    n + 1, 1, figsize=(2.5, fig_h),
    height_ratios=heights, gridspec_kw={"hspace": 0.105}, sharex=True,
)

# profile
ax_prof = axes[0]
xx = np.arange(total_bins)
for label in labels:
    ax_prof.plot(xx, profiles[label], color=colors[label], lw=1)
ax_prof.set_ylabel("Mean LOS\nresidual")
ax_prof.margins(x=0)
for xb in (tss_pos, tes_pos):
    ax_prof.axvline(xb, color="grey", lw=0.6, ls="--")

# heatmaps
im = None
for i, label in enumerate(labels):
    ax = axes[i + 1]
    im = ax.imshow(
        stacks[label], aspect="auto", cmap="magma",
        vmin=vmin, vmax=vmax, interpolation="none",
        extent=[0, total_bins, 0, len(stacks[label])],
    )
    ax.set_ylabel(f"{label}\n(n={len(stacks[label])})",
                  rotation=0, ha="right", va="center", fontsize=9)
    ax.set_yticks([])
    for xb in (tss_pos, tes_pos):
        ax.axvline(xb, color="white", lw=0.2, ls="--", alpha=0.7)
    for spine in ax.spines.values():
        spine.set_edgecolor(colors[label]); spine.set_linewidth(2); spine.set_visible(True)
    if i < n - 1:
        ax.tick_params(labelbottom=False)

# x ticks: TSS / TES labels
axes[-1].set_xticks([0, tss_pos, tes_pos, total_bins])
axes[-1].set_xticklabels([f"-{upstream//1000}kb", "TSS", "TES",
                          f"+{downstream//1000}kb"])
for spine in ax.spines.values():
    spine.set_linewidth(1.3)
ax.tick_params(width=1.3, length=2)

# shared colorbar
fig.subplots_adjust(right=0.86)
cax = fig.add_axes([0.88, 0.15, 0.05, 0.6])
fig.colorbar(im, cax=cax, label="LOS residual")

plt.savefig("gene_scaled_stackups_polII_Act23_new_regions2.svg", bbox_inches="tight")
plt.close(fig)

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

In [18]:
#### Fig. 6d

import bbi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42 
matplotlib.rcParams['svg.fonttype'] = 'none'

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],   
    'font.size': 9,           
    'axes.labelsize': 9,    
    'xtick.labelsize': 7,      
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.titlesize': 9,
})

######## config
BW_LOS = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_mega120mPD_DpnII_R12_20250129_5kb.bw"
BW_NEW = "/abyss/dlafonta/deeptools/CPM/HS37_20250528_5000.bw"

bed_sets = {   # label -> path
    "speckle": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_expressed_10kbfilter2.bed",
    "speckle_low": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_silent_10kbfilter2.bed",
    "Act23": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_InteriorAct23_expressed_10kbfilter2.bed",
    "Act23_low": "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_InteriorAct23_silent_10kbfilter2.bed",
}
colors = {
    "speckle":     "#9E0142",
    "speckle_low": "#F1A9A0",
    "Act23":       "#F46D43",
    "Act23_low":   "#FDAE61",
}

upstream = downstream = 10000
body_bins, flank_bins = 50, 50
half = 5

total_bins = flank_bins + body_bins + flank_bins
tss_pos, tes_pos = flank_bins, flank_bins + body_bins


# Pre-filter each BED to chroms present in BOTH tracks.
# Region tables stay in chr-prefixed space; HS37 is queried with
# stripped names via strip_chr=True.

with bbi.open(BW_LOS) as f:
    los_chroms = set(f.chromsizes)                    # chr-prefixed
with bbi.open(BW_NEW) as f:
    new_chroms = {"chr" + c for c in f.chromsizes}    # normalize bare -> chr

common = los_chroms & new_chroms

regs = {}
for label, path in bed_sets.items():
    r = pd.read_csv(path, sep="\t", header=None,
                    names=["chrom", "start", "end", "enst", "ensg", "strand", "length"])
    r = r[r.chrom.isin(common)].reset_index(drop=True)
    regs[label] = r

def stackup_scaled(f, reg, strip_chr=False):
    reg = reg.copy()
    if strip_chr:
        reg["chrom"] = reg["chrom"].str.replace("^chr", "", regex=True)
    chrom, start, end = reg.chrom.values, reg.start.values, reg.end.values
    is_plus = reg.strand.values == "+"

    up_start = np.where(is_plus, start - upstream, end)
    up_end   = np.where(is_plus, start,            end + upstream)
    up = f.stackup(chrom, up_start, up_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    body = f.stackup(chrom, start, end, bins=body_bins, missing=np.nan, oob=np.nan)

    dn_start = np.where(is_plus, end,              start - downstream)
    dn_end   = np.where(is_plus, end + downstream, start)
    dn = f.stackup(chrom, dn_start, dn_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    minus = ~is_plus
    up[minus]   = up[minus][:, ::-1]
    body[minus] = body[minus][:, ::-1]
    dn[minus]   = dn[minus][:, ::-1]
    return np.hstack([up, body, dn])


# PASS 1: LOS track — determine valid rows + sort order

orders, valids, los_stacks = {}, {}, {}
with bbi.open(BW_LOS) as f:
    for label in bed_sets:
        s = stackup_scaled(f, regs[label], strip_chr=False)

        valid = ~np.isnan(s).all(axis=1)
        s = s[valid]

        t0, t1 = max(0, tss_pos - half), tss_pos + half
        with np.errstate(invalid="ignore"):
            key = np.nanmean(s[:, t0:t1], axis=1)
        key = np.where(np.isnan(key), -np.inf, key)
        order = np.argsort(key)[::-1]

        valids[label] = valid
        orders[label] = order
        los_stacks[label] = s[order]


# PASS 2: NEW track — apply SAME valid mask + order (strip_chr!)

def build_with_fixed_order(bw_path, strip_chr):
    out = {}
    with bbi.open(bw_path) as f:
        for label in bed_sets:
            s = stackup_scaled(f, regs[label], strip_chr=strip_chr)
            s = s[valids[label]]      # same rows dropped as LOS pass
            s = s[orders[label]]      # same order as LOS pass
            out[label] = s
    return out

new_stacks = build_with_fixed_order(BW_NEW, strip_chr=True)
profiles   = {l: np.nanmean(s, axis=0) for l, s in new_stacks.items()}

/tmp/ipykernel_1551950/3325330794.py:100: RuntimeWarning: Mean of empty slice
  key = np.nanmean(s[:, t0:t1], axis=1)


In [19]:
# 1. repoint stacks/profiles to the NEW track
stacks   = new_stacks
profiles = {l: np.nanmean(s, axis=0) for l, s in stacks.items()}

# 2. recompute the color scale from the NEW data
allvals = np.concatenate([s[np.isfinite(s)].ravel() for s in stacks.values()])
vmin = 0
vmax = np.nanpercentile(allvals, 99)

###### PLot figure

labels = list(stacks.keys())
n = len(labels)
row_counts = np.array([len(stacks[l]) for l in labels], dtype=float)
rel = np.maximum(row_counts / row_counts.max(), 0.15)
heights = [0.64] + list(rel)
fig_h = 2 + 2.2 * n

fig, axes = plt.subplots(
    n + 1, 1, figsize=(2.5, fig_h),
    height_ratios=heights, gridspec_kw={"hspace": 0.105}, sharex=True,
)

# profile
ax_prof = axes[0]
xx = np.arange(total_bins)
for label in labels:
    ax_prof.plot(xx, profiles[label], color=colors[label], lw=1)
ax_prof.set_ylabel("Mean LOS\nresidual")
ax_prof.margins(x=0)
for xb in (tss_pos, tes_pos):
    ax_prof.axvline(xb, color="grey", lw=0.6, ls="--")

# heatmaps
im = None
for i, label in enumerate(labels):
    ax = axes[i + 1]
    im = ax.imshow(
        stacks[label], aspect="auto", cmap="magma",
        vmin=vmin, vmax=vmax, interpolation="none",
        extent=[0, total_bins, 0, len(stacks[label])],
    )
    ax.set_ylabel(f"{label}\n(n={len(stacks[label])})",
                  rotation=0, ha="right", va="center", fontsize=9)
    ax.set_yticks([])
    for xb in (tss_pos, tes_pos):
        ax.axvline(xb, color="white", lw=0.2, ls="--", alpha=0.7)
    for spine in ax.spines.values():
        spine.set_edgecolor(colors[label]); spine.set_linewidth(2); spine.set_visible(True)
    if i < n - 1:
        ax.tick_params(labelbottom=False)

# x ticks: TSS / TES labels
axes[-1].set_xticks([0, tss_pos, tes_pos, total_bins])
axes[-1].set_xticklabels([f"-{upstream//1000}kb", "TSS", "TES",
                          f"+{downstream//1000}kb"])
for spine in ax.spines.values():
    spine.set_linewidth(1.3)
ax.tick_params(width=1.3, length=2)

# shared colorbar
fig.subplots_adjust(right=0.86)
cax = fig.add_axes([0.88, 0.15, 0.05, 0.6])
fig.colorbar(im, cax=cax, label="LOS residual")

plt.savefig("gene_scaled_stackups_SLAM_Act23_new_regions2.svg", bbox_inches="tight")
plt.close(fig)

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

In [8]:
##### Fig. 6f

import bbi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42 
matplotlib.rcParams['svg.fonttype'] = 'none' 

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],  
    'font.size': 9,    
    'axes.labelsize': 9,      
    'xtick.labelsize': 7,      
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.titlesize': 9,
})

####### config
BW_A = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_PIIBNT240mPD_DpnII_R1_20200103_5kb.bw"
BW_B = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_PIIBTD240mPD_DpnII_R1_20200103_5kb.bw"

track_labels = {"A": "PIIB-NT", "B": "PIIB-TD"}
track_colors = {"A": "#9E0142", "B": "#3288BD"}

BED = "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_expressed_10kbfilter2.bed"

####### scaled-mode geometry
upstream = downstream = 10000
body_bins, flank_bins = 50, 50
total_bins = flank_bins + body_bins + flank_bins
tss_pos, tes_pos = flank_bins, flank_bins + body_bins

def stackup_scaled(f, reg):
    chrom, start, end = reg.chrom.values, reg.start.values, reg.end.values
    is_plus = reg.strand.values == "+"

    up_start = np.where(is_plus, start - upstream, end)
    up_end   = np.where(is_plus, start,            end + upstream)
    up = f.stackup(chrom, up_start, up_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    body = f.stackup(chrom, start, end, bins=body_bins, missing=np.nan, oob=np.nan)

    dn_start = np.where(is_plus, end,              start - downstream)
    dn_end   = np.where(is_plus, end + downstream, start)
    dn = f.stackup(chrom, dn_start, dn_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    minus = ~is_plus
    up[minus]   = up[minus][:, ::-1]
    body[minus] = body[minus][:, ::-1]
    dn[minus]   = dn[minus][:, ::-1]
    return np.hstack([up, body, dn])

###### region (filter to chroms in both tracks)
with bbi.open(BW_A) as f: chroms_a = set(f.chromsizes)
with bbi.open(BW_B) as f: chroms_b = set(f.chromsizes)
common = chroms_a & chroms_b

reg = pd.read_csv(BED, sep="\t", header=None,
                  names=["chrom","start","end","enst","ensg","strand","length"])
reg = reg[reg.chrom.isin(common)].reset_index(drop=True)

###### build profiles (mean across genes, ignoring NaN)
profiles = {}
with bbi.open(BW_A) as f:
    profiles["A"] = np.nanmean(stackup_scaled(f, reg), axis=0)
with bbi.open(BW_B) as f:
    profiles["B"] = np.nanmean(stackup_scaled(f, reg), axis=0)

###### figure: single overlaid line plot ---
fig, ax = plt.subplots(figsize=(5, 5))
xx = np.arange(total_bins)

for k in ("A", "B"):
    ax.plot(xx, profiles[k], color=track_colors[k], lw=2, label=track_labels[k])

for xb in (tss_pos, tes_pos):
    ax.axvline(xb, color="grey", lw=0.6, ls=":")

ax.set_ylabel("Mean LOS residual")
ax.set_ylim(-0.10, 0.025)
ax.set_xticks([0, tss_pos, tes_pos, total_bins])
ax.set_xticklabels([f"-{upstream//1000}kb", "TSS", "TES", f"+{downstream//1000}kb"])
ax.margins(x=0)
ax.legend(frameon=False)
ax.set_box_aspect(1)

plt.tight_layout()
plt.savefig("gene_scaled_profiles_TXblock2.svg", bbox_inches="tight")
plt.close(fig)

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

In [9]:
##### Fig. 6g


import bbi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42 
matplotlib.rcParams['svg.fonttype'] = 'none' 

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'font.size': 9,             
    'axes.labelsize': 9,      
    'xtick.labelsize': 7,        
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.titlesize': 9,
})

######## config
BW_A = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_contAMO120mPD_DpnII_20240828_5kb.bw"
BW_B = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_U1AMO120mPD_DpnII_20240828_5kb.bw"

track_labels = {"A": "contAMO", "B": "U1AMO"}
track_colors = {"A": "#9E0142", "B": "#3288BD"}

BED = "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_expressed_10kbfilter2.bed"

######## scaled-mode geometry 
upstream = downstream = 10000
body_bins, flank_bins = 50, 50
total_bins = flank_bins + body_bins + flank_bins
tss_pos, tes_pos = flank_bins, flank_bins + body_bins

def stackup_scaled(f, reg):
    chrom, start, end = reg.chrom.values, reg.start.values, reg.end.values
    is_plus = reg.strand.values == "+"

    up_start = np.where(is_plus, start - upstream, end)
    up_end   = np.where(is_plus, start,            end + upstream)
    up = f.stackup(chrom, up_start, up_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    body = f.stackup(chrom, start, end, bins=body_bins, missing=np.nan, oob=np.nan)

    dn_start = np.where(is_plus, end,              start - downstream)
    dn_end   = np.where(is_plus, end + downstream, start)
    dn = f.stackup(chrom, dn_start, dn_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    minus = ~is_plus
    up[minus]   = up[minus][:, ::-1]
    body[minus] = body[minus][:, ::-1]
    dn[minus]   = dn[minus][:, ::-1]
    return np.hstack([up, body, dn])

######## region (filter to chroms in both tracks)
with bbi.open(BW_A) as f: chroms_a = set(f.chromsizes)
with bbi.open(BW_B) as f: chroms_b = set(f.chromsizes)
common = chroms_a & chroms_b

reg = pd.read_csv(BED, sep="\t", header=None,
                  names=["chrom","start","end","enst","ensg","strand","length"])
reg = reg[reg.chrom.isin(common)].reset_index(drop=True)

######## build profiles (mean across genes, ignoring NaN)
profiles = {}
with bbi.open(BW_A) as f:
    profiles["A"] = np.nanmean(stackup_scaled(f, reg), axis=0)
with bbi.open(BW_B) as f:
    profiles["B"] = np.nanmean(stackup_scaled(f, reg), axis=0)

####### figure: single overlaid line plot
fig, ax = plt.subplots(figsize=(5, 5))
xx = np.arange(total_bins)

for k in ("A", "B"):
    ax.plot(xx, profiles[k], color=track_colors[k], lw=2, label=track_labels[k])

#ax.axhline(0, color="grey", lw=0.6, ls="--")
for xb in (tss_pos, tes_pos):
    ax.axvline(xb, color="grey", lw=0.6, ls=":")

ax.set_ylabel("Mean LOS residual")
ax.set_ylim(-0.10, 0.025)
ax.set_xticks([0, tss_pos, tes_pos, total_bins])
ax.set_xticklabels([f"-{upstream//1000}kb", "TSS", "TES", f"+{downstream//1000}kb"])
ax.margins(x=0)
ax.legend(frameon=False)
ax.set_box_aspect(1)

plt.tight_layout()
plt.savefig("gene_scaled_profiles_U14h2.svg", bbox_inches="tight")
#plt.savefig("gene_scaled_profiles_U14h.png", dpi=200, bbox_inches="tight")
plt.close(fig)

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

In [10]:
##### Fig. 6h

import bbi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42 
matplotlib.rcParams['svg.fonttype'] = 'none'

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'], 
    'font.size': 9,           
    'axes.labelsize': 9,     
    'xtick.labelsize': 7,        
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.titlesize': 9,
})

######## config
BW_A = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_RNaseIN4hPD_DpnII_R2_20230628_5kb.bw"
BW_B = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_RNaseA4hPD_DpnII_R2_20230628_5kb.bw"

track_labels = {"A": "RNaseIN", "B": "RNaseA"}
track_colors = {"A": "#9E0142", "B": "#3288BD"}

BED = "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_expressed_10kbfilter2.bed"

######## scaled-mode geometry
upstream = downstream = 10000
body_bins, flank_bins = 50, 50
total_bins = flank_bins + body_bins + flank_bins
tss_pos, tes_pos = flank_bins, flank_bins + body_bins

def stackup_scaled(f, reg):
    chrom, start, end = reg.chrom.values, reg.start.values, reg.end.values
    is_plus = reg.strand.values == "+"

    up_start = np.where(is_plus, start - upstream, end)
    up_end   = np.where(is_plus, start,            end + upstream)
    up = f.stackup(chrom, up_start, up_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    body = f.stackup(chrom, start, end, bins=body_bins, missing=np.nan, oob=np.nan)

    dn_start = np.where(is_plus, end,              start - downstream)
    dn_end   = np.where(is_plus, end + downstream, start)
    dn = f.stackup(chrom, dn_start, dn_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    minus = ~is_plus
    up[minus]   = up[minus][:, ::-1]
    body[minus] = body[minus][:, ::-1]
    dn[minus]   = dn[minus][:, ::-1]
    return np.hstack([up, body, dn])

######## region (filter to chroms in both tracks)
with bbi.open(BW_A) as f: chroms_a = set(f.chromsizes)
with bbi.open(BW_B) as f: chroms_b = set(f.chromsizes)
common = chroms_a & chroms_b

reg = pd.read_csv(BED, sep="\t", header=None,
                  names=["chrom","start","end","enst","ensg","strand","length"])
reg = reg[reg.chrom.isin(common)].reset_index(drop=True)

######## build profiles (mean across genes, ignoring NaN)
profiles = {}
with bbi.open(BW_A) as f:
    s = stackup_scaled(f, reg)
    s[~np.isfinite(s)] = np.nan          # inf/-inf -> nan
    profiles["A"] = np.nanmean(s, axis=0)
with bbi.open(BW_B) as f:
    s = stackup_scaled(f, reg)
    s[~np.isfinite(s)] = np.nan          # inf/-inf -> nan
    profiles["B"] = np.nanmean(s, axis=0)

######## figure: single overlaid line plot
fig, ax = plt.subplots(figsize=(5, 5))
xx = np.arange(total_bins)

for k in ("A", "B"):
    ax.plot(xx, profiles[k], color=track_colors[k], lw=2, label=track_labels[k])

for xb in (tss_pos, tes_pos):
    ax.axvline(xb, color="grey", lw=0.6, ls=":")

ax.set_ylabel("Mean LOS residual")
ax.set_ylim(-0.10, 0.025)
ax.set_xticks([0, tss_pos, tes_pos, total_bins])
ax.set_xticklabels([f"-{upstream//1000}kb", "TSS", "TES", f"+{downstream//1000}kb"])
ax.margins(x=0)
ax.legend(frameon=False)
ax.set_box_aspect(1)

plt.tight_layout()
plt.savefig("gene_scaled_profiles_RNase2.svg", bbox_inches="tight")
plt.close(fig)

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

In [11]:
##### Fig. 6i

import bbi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42 
matplotlib.rcParams['svg.fonttype'] = 'none' 

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],   
    'font.size': 9,           
    'axes.labelsize': 9,       
    'xtick.labelsize': 7,        
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.titlesize': 9,
})

######## config
BW_A = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562SpeckledTAG_Neg120mPD_DpnII_20231218_5kb.bw"
BW_B = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562SpeckledTAG_dTAG120mPD_DpnII_20231218_5kb.bw"

track_labels = {"A": "Neg", "B": "dTAG"}
track_colors = {"A": "#9E0142", "B": "#3288BD"}

BED = "/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_expressed_10kbfilter2.bed"

######## scaled-mode geometry
upstream = downstream = 10000
body_bins, flank_bins = 50, 50
total_bins = flank_bins + body_bins + flank_bins
tss_pos, tes_pos = flank_bins, flank_bins + body_bins

def stackup_scaled(f, reg):
    chrom, start, end = reg.chrom.values, reg.start.values, reg.end.values
    is_plus = reg.strand.values == "+"

    up_start = np.where(is_plus, start - upstream, end)
    up_end   = np.where(is_plus, start,            end + upstream)
    up = f.stackup(chrom, up_start, up_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    body = f.stackup(chrom, start, end, bins=body_bins, missing=np.nan, oob=np.nan)

    dn_start = np.where(is_plus, end,              start - downstream)
    dn_end   = np.where(is_plus, end + downstream, start)
    dn = f.stackup(chrom, dn_start, dn_end, bins=flank_bins, missing=np.nan, oob=np.nan)

    minus = ~is_plus
    up[minus]   = up[minus][:, ::-1]
    body[minus] = body[minus][:, ::-1]
    dn[minus]   = dn[minus][:, ::-1]
    return np.hstack([up, body, dn])

######## region (filter to chroms in both tracks)
with bbi.open(BW_A) as f: chroms_a = set(f.chromsizes)
with bbi.open(BW_B) as f: chroms_b = set(f.chromsizes)
common = chroms_a & chroms_b

reg = pd.read_csv(BED, sep="\t", header=None,
                  names=["chrom","start","end","enst","ensg","strand","length"])
reg = reg[reg.chrom.isin(common)].reset_index(drop=True)

######## build profiles (mean across genes, ignoring NaN)
profiles = {}
with bbi.open(BW_A) as f:
    profiles["A"] = np.nanmean(stackup_scaled(f, reg), axis=0)
with bbi.open(BW_B) as f:
    profiles["B"] = np.nanmean(stackup_scaled(f, reg), axis=0)

######## figure: single overlaid line plot
fig, ax = plt.subplots(figsize=(5, 5))
xx = np.arange(total_bins)

for k in ("A", "B"):
    ax.plot(xx, profiles[k], color=track_colors[k], lw=2, label=track_labels[k])

for xb in (tss_pos, tes_pos):
    ax.axvline(xb, color="grey", lw=0.6, ls=":")

ax.set_ylabel("Mean LOS residual")
ax.set_ylim(-0.10, 0.025)
ax.set_xticks([0, tss_pos, tes_pos, total_bins])
ax.set_xticklabels([f"-{upstream//1000}kb", "TSS", "TES", f"+{downstream//1000}kb"])
ax.margins(x=0)
ax.legend(frameon=False)
ax.set_box_aspect(1)

plt.tight_layout()
plt.savefig("gene_scaled_profiles_SpdTAG2.svg", bbox_inches="tight")
plt.close(fig)

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because